In [ ]:
import open3d as o3d
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from scipy.interpolate import CubicSpline

# Use interactive backend
matplotlib.use('TkAgg')

# Step 1: Load STL and Sample Points on the Surface
def load_stl_sample_surface(stl_file, num_points=50000):
    mesh = o3d.io.read_triangle_mesh(stl_file)
    mesh.compute_vertex_normals()
    
    # Sample points uniformly from the surface
    pcd = mesh.sample_points_uniformly(number_of_points=num_points)

    if len(pcd.points) < 2:
        raise ValueError("Error: Not enough points for path extraction.")

    points = np.asarray(pcd.points)
    
    return points

# Step 2: Organize Points into a Structured Path
def create_structured_path(points, axis=2, num_layers=50):
    """
    Organizes sampled points into a structured meandering path.
    - axis=2 means layers are sorted along the Z-axis.
    - num_layers determines how many layers we divide the model into.
    """
    min_val, max_val = np.min(points[:, axis]), np.max(points[:, axis])
    layer_intervals = np.linspace(min_val, max_val, num_layers)

    structured_path = []

    for i in range(len(layer_intervals) - 1):
        # Select points within the current Z-layer
        layer_points = points[
            (points[:, axis] >= layer_intervals[i]) & (points[:, axis] < layer_intervals[i + 1])
        ]

        if len(layer_points) == 0:
            continue

        # Sort points within the layer for a continuous scan
        sorted_layer = layer_points[np.argsort(layer_points[:, (axis + 1) % 3])]  # Sort along another axis
        if i % 2 == 1:
            sorted_layer = sorted_layer[::-1]  # Zigzag motion

        structured_path.append(sorted_layer)

    return np.vstack(structured_path)

# Step 3: Smooth Path using Cubic Spline
def smooth_path(path_points):
    if len(path_points) < 3:
        print("Warning: Not enough points for smoothing. Returning raw path.")
        return path_points

    t = np.linspace(0, 1, len(path_points))
    cs_x = CubicSpline(t, path_points[:, 0])
    cs_y = CubicSpline(t, path_points[:, 1])
    cs_z = CubicSpline(t, path_points[:, 2])

    t_fine = np.linspace(0, 1, 10 * len(path_points))  # Increase resolution
    smooth_path_points = np.vstack((cs_x(t_fine), cs_y(t_fine), cs_z(t_fine))).T
    return smooth_path_points

# Step 4: Animate the Robot Path Along the Model Surface
def animate_path(smoothed_points):
    fig = plt.figure(figsize=(8, 6))
    ax = fig.add_subplot(111, projection='3d')

    ax.set_xlim([np.min(smoothed_points[:, 0]), np.max(smoothed_points[:, 0])])
    ax.set_ylim([np.min(smoothed_points[:, 1]), np.max(smoothed_points[:, 1])])
    ax.set_zlim([np.min(smoothed_points[:, 2]), np.max(smoothed_points[:, 2])])

    ax.set_xlabel("X Axis")
    ax.set_ylabel("Y Axis")
    ax.set_zlabel("Z Axis")
    ax.set_title("Robot Path Along Model Surface")

    # Plot the full path in the background
    ax.plot(smoothed_points[:, 0], smoothed_points[:, 1], smoothed_points[:, 2], 'gray', alpha=0.3)

    # Initialize animated point
    point, = ax.plot([], [], [], 'ro', markersize=8, label="Robot Position")

    def update(frame):
        point.set_data([smoothed_points[frame, 0]], [smoothed_points[frame, 1]])
        point.set_3d_properties([smoothed_points[frame, 2]])
        return point,

    ani = animation.FuncAnimation(fig, update, frames=len(smoothed_points), interval=30, blit=False)

    plt.legend()
    plt.show(block=True)  # Keep the animation open

# Main Execution
stl_file = "your_model.stl"  # Replace with your STL file

try:
    points = load_stl_sample_surface(stl_file, num_points=50000)

    # Create a structured scanning path
    structured_path = create_structured_path(points, axis=2, num_layers=50)
    smoothed_path = smooth_path(structured_path)

    # Run Animation
    animate_path(smoothed_path)

except ValueError as e:
    print(e)
